In [9]:
import pandas as pd 
import numpy as np 
from pathlib import Path

ROOT = Path("../..")
RAW = ROOT / "data" / "raw" / "qualita_aria_campania_validati.csv"
PROCESSED = ROOT / "data" / "processed"

RAW.exists()
df = pd.read_csv(RAW, dtype=str, encoding="utf-8-sig")
#df.head(10)
df.tail()

,_id,Stazione,Descrizione,Inquinante,Data_ora,Valore,Um
4905163,99580,IT2283A,Ottati Alburni,SO2,26-10-2026 23:00:00 +01:00,4.71,µg/m3
4905164,99604,IT2342A,S. Gregorio Matese Lago,NO2,26-10-2026 23:00:00 +01:00,0,µg/m3
4905165,99628,IT2342A,S. Gregorio Matese Lago,O3,26-10-2026 23:00:00 +01:00,61.2,µg/m3
4905166,99652,IT2342A,S. Gregorio Matese Lago,PM10,26-10-2026 23:00:00 +01:00,2.08,µg/m3
4905167,99676,IT2342A,S. Gregorio Matese Lago,"PM2,5",26-10-2026 23:00:00 +01:00,1.22,µg/m3


La colonna _id è un numero di riga interno a ciascun file mensile (riparte da 1 ogni mese) e non identifica le misure

In [33]:
df.shape
#df["Stazione"].unique()
df["Stazione"].nunique()
#df["Stazione"].value_counts()
df["Inquinante"].value_counts()
df.isna().sum()
df["_id"].nunique()
#len(df)

#nb_id_uniques = df[df["Stazione"] == "IT2283A"]["_id"].nunique()
#print(nb_id_uniques)

df.groupby("Inquinante")["Um"].unique()


Inquinante
C6H6     [µg/m3]
CO       [mg/m3]
NO2      [µg/m3]
O3       [µg/m3]
PM10     [µg/m3]
PM2,5    [µg/m3]
SO2      [µg/m3]
Name: Um, dtype: object

abbiamo cancellato la colonna _id e abbiamo convertito la colonna data che era string nel formato datetime

In [ ]:
df = df.drop(columns=["_id"], errors="ignore")
#print(df.head())
forme = df["Data_ora"].str.replace(r"\d", "9", regex=True)
forme.value_counts()

s = df["Data_ora"].str.strip()
dt1 = pd.to_datetime(s, format="%d-%m-%Y %H:%M:%S %z", errors="coerce", utc=True)
dt1.isna().sum()


s2 = s.str.replace(r"([+-]\d{2})$", r"\1:00", regex=True)
dt2 = pd.to_datetime(s2, format="%Y-%m-%d %H:%M:%S%z", errors="coerce", utc=True)
dt2.isna().sum()

dt = dt1.fillna(dt2) # dt lista di datetime, dove dt1 è vuoto prende i valori di dt2
dt.isna().sum()

np.int64(0)

abbiamo fatto la conversione di "valore" da str a numero

In [ ]:
df["Valore"] = pd.to_numeric(df["Valore"].str.replace(",", "."), errors="coerce")
df["Valore"].isna().sum()

in valore min abbiamo -9999 in max 1437008 : non va bene

analisi dei valori strani

In [86]:
#df["Data_ora"] = dt.dt.tz_convert("Etc/GMT-1").dt.tz_localize(None)
#del s, s2, dt1, dt2, dt
df.head()
#print(df.iloc[4200])
#print(df.dtypes)
df["Data_ora"].max()
df["Valore"].min()


#print(sum(df["Valore"] == -9999.0))
ls = df[(df["Valore"] < 0) & (df["Valore"] != -9999.0)]
#print(ls)
#print(df[df["Valore"] != -9999].groupby("Inquinante")["Valore"].describe())
print(df.nlargest(10, "Valore"))

        Stazione                 Descrizione Inquinante            Data_ora  \
661220   IT1504A  SA22 Ospedale via Vernieri       PM10 2023-09-01 09:00:00   
3059863  IT0934A           BN32 Via Mustilli      PM2,5 2025-06-13 12:00:00   
3059503  IT0934A           BN32 Via Mustilli      PM2,5 2025-06-13 10:00:00   
2400918  IT1493A        NA09 I.T.I.S. ARGINE        NO2 2024-11-22 23:00:00   
3634922  IT2226A  Pomigliano d'Arco Area Asi       PM10 2026-01-01 12:00:00   
2562812  IT0898A        NA06 MUSEO NAZIONALE       PM10 2025-01-01 01:00:00   
2161686  IT1486A       CE51 ISTITUTO MANZONI       PM10 2024-09-27 00:00:00   
2161866  IT1486A       CE51 ISTITUTO MANZONI       PM10 2024-09-27 01:00:00   
2162046  IT1486A       CE51 ISTITUTO MANZONI       PM10 2024-09-27 02:00:00   
2162226  IT1486A       CE51 ISTITUTO MANZONI       PM10 2024-09-27 03:00:00   

            Valore     Um  
661220   1437008.0  µg/m3  
3059863   101090.0  µg/m3  
3059503    17792.5  µg/m3  
2400918     4992.6

piccoli analisi sui picci di rilevamento strani come ferragosto a napoli, e messa di NaN sui valori negativi

In [97]:
masque = (
    (df["Stazione"] == "IT1504A")
    & (df["Inquinante"] == "PM10")
    & (df["Data_ora"].between("2023-09-01 03:00", "2023-09-01 15:00"))
)
df[masque][["Data_ora", "Valore"]]

masque = (
    (df["Stazione"] == "IT0898A")
    & (df["Inquinante"] == "PM10")
    & (df["Data_ora"].between("2024-12-31 18:00", "2025-01-01 10:00"))
)
df[masque][["Data_ora", "Valore"]]

masque = (
    (df["Stazione"] == "IT1486A")
    & (df["Inquinante"] == "PM10")
    & (df["Data_ora"].between("2024-09-26 00:00", "2024-09-26 23:00"))
)
df[masque][["Data_ora", "Valore"]]

df.loc[df["Valore"] < 0, "Valore"] = np.nan
print("Valori mancanti (NaN):", df["Valore"].isna().sum())

df.loc[df["Valore"] < 0, "Valore"] = np.nan
print("Valori mancanti (NaN):", df["Valore"].isna().sum())

Valori mancanti (NaN): 202527
Valori mancanti (NaN): 202527


il numero di giorni dove il sensore a catturato la stessa cifra tutta la giornata per inquinante

In [99]:
df["giorno"] = df["Data_ora"].dt.normalize()

stat = df.groupby(["Stazione", "Inquinante", "giorno"])["Valore"].agg(["count", "nunique", "mean"])
bloccati = stat[(stat["count"] >= 18) & (stat["nunique"] == 1)]

print("Giorni con valore costante:", len(bloccati))
bloccati.groupby("Inquinante").size()
bloccati.sort_values("mean", ascending=False).head(20)

Giorni con valore costante: 13427


count  nunique    mean
Stazione Inquinante giorno                            
IT1486A  PM10       2024-09-27     24        1  2453.0
                    2024-09-24     24        1  2213.5
                    2024-08-31     24        1  2083.0
                    2024-09-13     24        1  1688.0
                    2024-09-26     24        1  1596.0
IT2277A  PM10       2023-09-22     24        1   583.2
IT2283A  PM10       2024-03-31     23        1   260.5
IT1495A  PM10       2025-01-01     24        1   200.0
                    2024-12-31     24        1   200.0
IT2283A  PM10       2025-03-15     24        1   184.4
IT1495A  PM10       2024-03-31     23        1   183.2
                    2024-04-01     24        1   181.9
IT2211A  PM10       2025-01-01     24        1   173.2
IT2283A  PM10       2024-04-01     24        1   167.4
IT2268A  PM10       2024-04-01     24        1   162.9
IT2211A  PM10       2024-06-21     24        1   160.9
IT1487A  PM10       2024-04-01     24        1   142.5
IT2283A  PM10       2023-10-24     24        1   141.7
                    2024-09-08     24        1   137.1
IT1487A  PM10       2024-03-31     23        1   136.3

Sono state escluse:

le stazioni non fisse (laboratori mobili LM/MMA/MMB, impianti STIR, EDI), presenti solo dal 7 al 18 ottobre 2025;
le stazioni installate a marzo 2026 (IT2220A, IT2222A, IT2224A, IT2225A, IT2342A), con circa 7 mesi di dati: senza un inverno completo, i loro valori medi non sarebbero confrontabili con quelli delle altre stazioni;
la stazione IT2269A (Salerno Parco Mercatello), dismessa ad agosto 2023, con meno di 5 mesi di dati.

In [ ]:
# Stazioni escluse (motivazione nel report)
nuove_2026 = ["IT2220A", "IT2222A", "IT2224A", "IT2225A", "IT2342A"]
dismesse = ["IT2269A"]

fisse = df["Stazione"].str.match(r"^IT\d{4}A$")
da_tenere = fisse & ~df["Stazione"].isin(nuove_2026 + dismesse)

avant = len(df)
df = df[da_tenere]
print("Righe eliminate:", avant - len(df))
print("Stazioni rimaste:", df["Stazione"].nunique())


Righe eliminate: 0
Stazioni rimaste: 36


,Stazione,Descrizione,Inquinante,Data_ora,Valore,Um,giorno
0,IT2283A,OTTATI - ALBURNI,PM10,2023-03-20,18.3,µg/m3,2023-03-20
1,IT2227A,AVELLINO - SCUOLA ALIGHIERI,PM10,2023-03-20,78.2,µg/m3,2023-03-20
2,IT2226A,Pomigliano d'Arco Area ASI,PM10,2023-03-20,31.6,µg/m3,2023-03-20
3,IT0936A,AV41 SCUOLA V CIRCOLO,PM10,2023-03-20,73.8,µg/m3,2023-03-20
4,IT2228A,BENEVENTO CAMPO SPORTIVO,PM10,2023-03-20,40.1,µg/m3,2023-03-20


verifica dell'andamento dei picci possitivi strani

In [112]:

masque = (
    (df["Stazione"] == "IT1493A")
    & (df["Inquinante"] == "NO2")
    & (df["Data_ora"].between("2024-11-22 17:00", "2024-11-23 05:00"))
)
df[masque][["Data_ora", "Valore"]]

masque = (
    (df["Stazione"] == "IT0934A")
    & (df["Inquinante"] == "PM2,5")
    & (df["Data_ora"].between("2025-06-13 00:00", "2025-06-13 23:00"))
)
df[masque][["Data_ora", "Valore"]]

masque = (
    (df["Stazione"] == "IT2226A")
    & (df["Inquinante"] == "PM10")
    & (df["Data_ora"].between("2025-12-31 18:00", "2026-01-01 18:00"))
)
df[masque][["Data_ora", "Valore"]]

,Data_ora,Valore
3631878,2025-12-31 18:00:00,8.4
3632048,2025-12-31 19:00:00,8.0
3632218,2025-12-31 20:00:00,NaN
3632388,2025-12-31 21:00:00,NaN
3632558,2025-12-31 22:00:00,18.7
3632728,2025-12-31 23:00:00,55.2
3632898,2026-01-01 00:00:00,335.8
3633067,2026-01-01 01:00:00,678.9
3633236,2026-01-01 02:00:00,793.9
3633405,2026-01-01 03:00:00,699.7


Valori aberranti impostati a NaN (valori isolati, circondati da valori normali o mancanti):

IT0934A, PM2,5, 13/06/2025 ore 10:00 e 12:00 (17.792,5 e 101.090 µg/m³)
IT1493A, NO2, 22/11/2024 ore 23:00 (4.992,6 µg/m³)
IT1504A, PM10, 01/09/2023 ore 09:00 (1.437.008 µg/m³), unico valore in una mattinata di dati mancanti

In [114]:
errori = (
    (df["Stazione"] == "IT0934A")
    & (df["Inquinante"] == "PM2,5")
    & (df["Data_ora"].isin(pd.to_datetime(["2025-06-13 10:00", "2025-06-13 12:00"])))
)
print("Valori da annullare:", errori.sum())
df.loc[errori, "Valore"] = np.nan

errori = (
    (df["Stazione"] == "IT1493A")
    & (df["Inquinante"] == "NO2")
    & (df["Data_ora"] == pd.Timestamp("2024-11-22 23:00"))
)
print("Valori da annullare:", errori.sum())
df.loc[errori, "Valore"] = np.nan

errori = (
    (df["Stazione"] == "IT1504A")
    & (df["Inquinante"] == "PM10")
    & (df["Data_ora"] == pd.Timestamp("2023-09-01 09:00"))
)
print("Valori da annullare:", errori.sum())
df.loc[errori, "Valore"] = np.nan

df.head()

Valori da annullare: 2
Valori da annullare: 1
Valori da annullare: 1


,Stazione,Descrizione,Inquinante,Data_ora,Valore,Um,giorno
0,IT2283A,OTTATI - ALBURNI,PM10,2023-03-20,18.3,µg/m3,2023-03-20
1,IT2227A,AVELLINO - SCUOLA ALIGHIERI,PM10,2023-03-20,78.2,µg/m3,2023-03-20
2,IT2226A,Pomigliano d'Arco Area ASI,PM10,2023-03-20,31.6,µg/m3,2023-03-20
3,IT0936A,AV41 SCUOLA V CIRCOLO,PM10,2023-03-20,73.8,µg/m3,2023-03-20
4,IT2228A,BENEVENTO CAMPO SPORTIVO,PM10,2023-03-20,40.1,µg/m3,2023-03-20


save

In [115]:
df = df.drop(columns=["giorno"], errors="ignore")
df.to_parquet(PROCESSED / "aria_campania_orario.parquet", index=False)
print("Righe salvate:", len(df))

Righe salvate: 4770326
